In [ ]:
import datetime

if not hasattr(datetime, "UTC"):
    datetime.UTC = datetime.timezone.utc

In [ ]:
! pip install lifelines

In [ ]:
# Imports here.
import numpy as np
import pandas as pd
import os
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import fdrcorrection
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#ndd_list = ['AD', 'DEM', 'PD', 'ALS', 'VAS']
ndd_list = ['AD', 'DEM', 'PD', 'VAS']
condition_list =['K11_ORAL', 'J10_INFLUPNEU', 'AB1_OTHER_BACTERIAL', 'K11_APPENDIX', 'AB1_BACTINF_NOS', 'J10_PNEUMOBACT', 'AB1_OTHER_SPIROCHAETAL', 'AB1_BACT_INTEST_OTH', 'O15_PUERP_SEPSIS', 'M13_REACTARTH', 'AB1_BACT_BIR_OTHER_INF_AGENTS', 'AB1_TUBERCULOSIS', 'M13_PYOGARTH', 'AB1_SALMONELLA_OTH', 'G6_MENINGBACT', 'AB1_SYPHILIS', 'AB1_SEQULAE_TUBERCU', 'AB1_ZOONOTIC_BACTERIAL', 'P16_BACTERIAL_SEPSIS_NEWBO', 'AB1_TYPHOIDPARATHYPHOID', 'AB1_ CHOLERA']
print(condition_list)
print(len(condition_list))
print(len(ndd_list))

In [ ]:
year = '2024'
date = 'JULY_1_2026'
model = 'all'
group_list = ['all']
lag_list =[0]

failed_codes = []
results = []
cov_list = []

for ndd in ndd_list:
    
    for lag in lag_list:

        for group in group_list:

            #Load df
            df = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/coxfiles/{ndd}_{date}_ready_cox.csv', parse_dates = True, low_memory = False)
            
            # Find codes to use so we don't have to use EVERYTHING
            codes_with_data = []

            for code in condition_list:

                m = df[['age_at_tenure', 'SEX', 'tenure', ndd, f'QC{lag}_{code}', 'APOE']]

                n=sum(m[f'QC{lag}_{code}'])
                df_pair = m[m[f'QC{lag}_{code}']==1]
                n_pairs = sum(df_pair[ndd])
                if n == 0:
                    pass
                elif n_pairs < 5:
                    pass
                elif n == n_pairs:
                    pass
                else:
                    print(code)
                    codes_with_data.append(code)

            print(ndd)
            print(len(codes_with_data))

            for code in codes_with_data:  
                
                try:
                    #m = df[df[f'{code}_exclude']==False]
                    m = df[['age_at_tenure', 'SEX', 'tenure', ndd, f'QC{lag}_{code}', 'APOE']]
                    
                    n=sum(m[f'QC{lag}_{code}'])
                    df_pair = m[m[f'QC{lag}_{code}']==1]
                    n_pairs = sum(df_pair[ndd])

                    formula=f"C(SEX) + C(QC{lag}_{code}) + C(APOE) + age_at_tenure"
                    cph = CoxPHFitter()
                    cph.fit(m, duration_col = 'tenure', event_col = ndd, formula = formula, fit_options = {'step_size':0.1}, show_progress=False)
                    #cph.print_summary()
                    #cph.plot()

                except ConvergenceError:
                    print(f"⚠️  Skipping {code}: model failed to converge.")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue

                except Exception as e:
                    print(f"⚠️  Skipping {code}: unexpected error -> {e}")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue
                    
                # Extract betas and SEs for a variable of interest
                beta = cph.params_[f'C(QC{lag}_{code})[T.1]']
                se = cph.standard_errors_[f'C(QC{lag}_{code})[T.1]']
    
                # extract HR, CI, p for each model
                covariate = code
                summary = cph.summary.loc[f'C(QC{lag}_{code})[T.1]']
                #print(summary)
                HR = summary['exp(coef)']
                ci_min = summary['exp(coef) lower 95%']
                ci_max = summary['exp(coef) upper 95%']
                p = summary['p'] 
    
                print(covariate, ndd, HR, beta, se, ci_min, ci_max, p, n_pairs, n)
                results.append((covariate, ndd, model, lag, HR, beta, se, ci_min, ci_max, p, n_pairs, n))
                
cox1 = pd.DataFrame(results, columns=('PRIOR','OUTCOME', 'MODEL', 'LAG', 'HR', 'beta', 'se', 'ci_min', "ci_max", 'P_VAL', "N_pairs", "N"))

In [ ]:
#Combine results
output = pd.concat([cox1])

#Adding FDR Correction

#Sort P-values
output = output.sort_values(by = "P_VAL")

#Drop Nan-values
output = output.dropna()

#FDR Correction
rejected, p_corr = fdrcorrection(output['P_VAL'], is_sorted=True)
output['P_CORR'] = p_corr
output['SIGNIFICANT'] = rejected

output

In [ ]:
output = output.to_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/output_5NDD.csv')

## Lags

In [ ]:
year = '2024'
date = 'JULY_1_2026'
model = 'all'
group_list = ['all']
lag_list = ['0', '0_1', '1_5', '5']

failed_codes = []
results = []
cov_list = []

for ndd in ndd_list:
    
    for lag in lag_list:

        for group in group_list:

            #Load df
            df = pd.read_csv(f'/home/jupyter/workspace/WORKSPACE_BUCKET/data/coxfiles/{ndd}_{date}_ready_cox.csv', parse_dates = True, low_memory = False)
            
            # Find codes to use so we don't have to use EVERYTHING
            codes_with_data = []

            for code in condition_list:

                m = df[['age_at_tenure', 'SEX', 'tenure', ndd, f'QC{lag}_{code}', 'APOE']]

                n=sum(m[f'QC{lag}_{code}'])
                df_pair = m[m[f'QC{lag}_{code}']==1]
                n_pairs = sum(df_pair[ndd])
                if n == 0:
                    pass
                elif n_pairs < 5:
                    pass
                elif n == n_pairs:
                    pass
                else:
                    print(code)
                    codes_with_data.append(code)

            print(ndd)
            print(len(codes_with_data))

            for code in codes_with_data:  
                
                try:
                    #m = df[df[f'{code}_exclude']==False]
                    m = df[['age_at_tenure', 'SEX', 'tenure', ndd, f'QC{lag}_{code}', 'APOE']]
                    
                    n=sum(m[f'QC{lag}_{code}'])
                    df_pair = m[m[f'QC{lag}_{code}']==1]
                    n_pairs = sum(df_pair[ndd])

                    formula=f"C(SEX) + C(QC{lag}_{code}) + C(APOE) + age_at_tenure"
                    cph = CoxPHFitter()
                    cph.fit(m, duration_col = 'tenure', event_col = ndd, formula = formula, fit_options = {'step_size':0.1}, show_progress=False)
                    #cph.print_summary()
                    #cph.plot()

                except ConvergenceError:
                    print(f"⚠️  Skipping {code}: model failed to converge.")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue

                except Exception as e:
                    print(f"⚠️  Skipping {code}: unexpected error -> {e}")
                    failed_codes.append(ndd)
                    failed_codes.append(code)
                    failed_codes.append(lag)
                    continue
                    
                # Extract betas and SEs for a variable of interest
                beta = cph.params_[f'C(QC{lag}_{code})[T.1]']
                se = cph.standard_errors_[f'C(QC{lag}_{code})[T.1]']
    
                # extract HR, CI, p for each model
                covariate = code
                summary = cph.summary.loc[f'C(QC{lag}_{code})[T.1]']
                #print(summary)
                HR = summary['exp(coef)']
                ci_min = summary['exp(coef) lower 95%']
                ci_max = summary['exp(coef) upper 95%']
                p = summary['p'] 
    
                print(covariate, ndd, HR, beta, se, ci_min, ci_max, p, n_pairs, n)
                results.append((covariate, ndd, model, lag, HR, beta, se, ci_min, ci_max, p, n_pairs, n))
                
cox2 = pd.DataFrame(results, columns=('PRIOR','OUTCOME', 'MODEL', 'LAG', 'HR', 'beta', 'se', 'ci_min', "ci_max", 'P_VAL', "N_pairs", "N"))

In [ ]:
#Combine results
output = pd.concat([cox2])

#Adding FDR Correction

#Sort P-values
output = output.sort_values(by = "P_VAL")

#Drop Nan-values
output = output.dropna()

#FDR Correction
rejected, p_corr = fdrcorrection(output['P_VAL'], is_sorted=True)
output['P_CORR'] = p_corr
output['SIGNIFICANT'] = rejected

output

In [ ]:
output = output.to_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/output_lags.csv')